In [ ]:

setwd(here::here("./10_data_analysis"))
knitr::opts_knit$set(root.dir = setwd(here::here("./10_data_analysis")))
getwd()
set.seed(666)

In [ ]:
options(scipen = 999)

# Save figures in specific place
knitr::opts_chunk$set(autodep        = TRUE,
                      cache          = FALSE,
                      cache.comments = TRUE,
                      comment = "",
                      # Save figures as pdf ?
                      dev = c( "png", "pdf"),
                  
                      # Include code?
                      echo           = TRUE,
                      error          = FALSE,
                      fig.align      = "center",
                      
                      # Path where figures are going to be store pdf single 
                      # figures
                      #Edit path in the next line and comment out this line   
                      #fig.path       = paste0("./666_figures", "/"),
                      fig.width      = 11,
                      fig.height     = 7,
                      message        = FALSE,
                      warning        = FALSE)

In [ ]:
library(dplyr)
library(tidyr)
library(purrr)
library(ggplot2)
library(patchwork)

# 1. Load data

In [ ]:
load("../01_euc_saligna/euc_saligna_processed_data/euc_saligna_rwc_data.RData")
load("../02_euc_grandis/euc_grandis_processed_data/euc_grandis_rwc_data.RData")
load("../03_euc_crebra/euc_crebra_processed_data/euc_crebra_rwc_data.RData")
load("../04_euc_dunnii/euc_dunnii_processed_data/euc_dunnii_rwc_data.RData")
load("../05_euc_tereticornis/euc_tereticornis_processed_data/euc_tereticornis_rwc_data.RData")
load("../06_euc_melliodora/euc_melliodora_processed_data/euc_melliodora_rwc_data.RData")
load("../07_euc_sideroxylum/euc_sideroxylum_processed_data/euc_sideroxylum_rwc_data.RData")
load("../08_euc_fibrosa/euc_fibrosa_processed_data/euc_fibrosa_rwc_data.RData")

# 2. Merge data

In [ ]:
droughtbox_conditions <-
    
    list(euc_saligna_rwc_data,
                  euc_grandis_rwc_data,
                  euc_crebra_rwc_data,
                  euc_melliodora_rwc_data,
                  euc_dunnii_rwc_data,
                  euc_tereticornis_rwc_data,
                  euc_sideroxylum_rwc_data,
                  euc_fibrosa_rwc_data) %>%
                 
                reduce(rbind) %>%
                ungroup() %>% 
    
                select(tc_avg_deg_c_avg,rh_avg_percent_avg, 
                       vpd_avg_kpa_avg, set_temperature, vpd_control) %>% 
                 
                group_by(set_temperature, vpd_control) %>% 
                
                summarise(median_vpd = median(vpd_avg_kpa_avg),
                            sd_vpd = sd(vpd_avg_kpa_avg),
                            median_rh = median(rh_avg_percent_avg),
                            sd_rh = sd(rh_avg_percent_avg),
                          .groups   = "drop") %>% 
    
                bind_rows(filter(., set_temperature == 25) %>%
                          mutate(vpd_control = "no")) %>% 
                arrange(set_temperature) %>% 
                
                mutate(label_trt = factor(if_else(vpd_control == "no", 
                                                  "dry_heat", "humid_heat"),
                                          levels = c("humid_heat", "dry_heat")),
                       
                       label  = factor(if_else(set_temperature == 25, "baseline", 
                                               as.character(label_trt)),
                                       levels = c("baseline", "humid_heat", 
                                                  "dry_heat")))


# 3. Figures

In [ ]:

plot_vpd <-
    ggplot(data = droughtbox_conditions, aes(x = factor(set_temperature), 
                                             y = median_vpd,
                                             color = label, 
                                             fill = label)) +
                        
            geom_hline(yintercept = 2, linetype = "dashed", color = "#E0E0E0") +
                        
            geom_errorbar(aes(ymin = median_vpd - sd_vpd, 
                              ymax = median_vpd + sd_vpd),
                          width = 0.3,
                          show.legend = FALSE,
                          colour = "black",
                          linewidth =  0.85) +
                
            geom_line(aes(group = vpd_control, colour = label_trt), 
                      linewidth = 2, show.legend = FALSE) +
                        
            geom_point(shape = 23, size = 9, stroke = 1.5, colour = "black") +
    
            theme_bw(base_size = 60) +
                
                        
            labs(x = 'Temperature (ºC)', y =  expression(VPD~(kPa))) +
                    
            ## Set colors                
            scale_fill_manual(values = c("baseline" = "#6E6E6E",
                                         "humid_heat" = "#293352",
                                         "dry_heat" = "#9e1523"),
                              labels = c("baseline" = "Baseline",
                                         "humid_heat" = "Humid heat",
                                         "dry_heat" = "Dry heat"),
                              
                              breaks = c("baseline", "humid_heat", "dry_heat")) + 
    
            scale_color_manual(values = c("baseline" = "#6E6E6E" ,
                                          "humid_heat" ="#293352",
                                          "dry_heat" = "#9e1523")) +
                
             theme(legend.position = "inside", legend.title = element_blank(),
                   legend.justification = c(0.005, 1.04),
                   legend.position.inside = c(0.005, 1.04),
                   legend.text = element_text(size = 30),
                   legend.background = element_blank(),
                   panel.grid.major = element_blank(),
                   panel.grid.minor = element_blank(),
                   strip.background = element_rect(fill = "white",colour = NA),
                   strip.text = element_text(colour = "black",face = "bold"),
                   panel.border = element_rect(colour = "black",fill = NA,
                                               linewidth = 4),
                   axis.text    = element_text(colour = "black"),
                   axis.ticks   = element_line(colour = "black")) + 
                    
            # Increase the tickmarks of the axes
            scale_y_continuous(breaks = seq(0, 14, 1)) +
            guides(fill = guide_legend(override.aes = list(shape = 23, size = 8,
                                                           stroke = 1.5)),
                   colour = "none") 




In [ ]:

plot_rh <- 

    ggplot(data = droughtbox_conditions, aes(x = factor(set_temperature), 
                                             y = median_rh,
                                             color = label, 
                                             fill = label)) +
    
        geom_errorbar(aes(ymin = median_rh - sd_rh, ymax = median_rh + sd_rh),
                      width = 0.3,
                      show.legend = FALSE,
                      colour = "black",
                      linewidth =  0.85) +
                        
        geom_line(aes(group = vpd_control, colour = label_trt), linewidth = 2, 
                  show.legend = FALSE) +
                        
        geom_point(shape = 23, size = 9, stroke = 1.5, colour = "black") +
    
        theme_bw(base_size = 60) +
    
        labs(x = 'Temperature (ºC)', y =  expression(`Relative Humidity`~(`%`))) + 
                        
        # Set colors                
        scale_fill_manual(values = c("baseline" = "#6E6E6E",
                                     "humid_heat" = "#293352",
                                     "dry_heat" = "#9e1523"),
                          
                          labels = c("baseline" = "Baseline",
                                     "humid_heat" = "Humid heat",
                                     "dry_heat" = "Dry heat"),
                          breaks = c("baseline", "humid_heat", "dry_heat")) + 
                    
        scale_color_manual(values = c("baseline" = "#6E6E6E" ,
                                      "humid_heat" ="#293352",
                                      "dry_heat" = "#9e1523")) +
        
        theme(legend.position = "inside", legend.title = element_blank(),
              legend.justification = c(0.005, 1.04),
              legend.position.inside = c(0.005, 1.04),
              legend.text = element_text(size = 30),
              legend.background = element_blank(),
              panel.grid.major = element_blank(),
              panel.grid.minor = element_blank(),
              strip.background = element_rect(fill = "white",colour = NA),
              strip.text = element_text(colour = "black",face = "bold"),
              panel.border = element_rect(colour = "black",fill = NA,
                                          linewidth = 4),
              axis.text    = element_text(colour = "black"),
              axis.ticks   = element_line(colour = "black")) + 
                    
            
        # Increase the tickmarks of the axes
        scale_y_continuous(breaks = seq(0, 100, 10)) +
        
        coord_cartesian(ylim = c(0, 100)) +
        guides(fill = guide_legend(override.aes = list(shape = 23, size = 8, 
                                                       stroke = 1.5)),
               colour = "none")


In [ ]:
plot_vpd + plot_rh  +  plot_annotation(tag_levels = 'A')

In [ ]:
droughtbox_conditions %>% 
    group_by(set_temperature, vpd_control)  %>% 
    select(median_rh,median_vpd) %>% 
    arrange(vpd_control)
    

In [ ]:
droughtbox_conditions %>% 
    group_by(set_temperature, vpd_control)  %>% 
    select(median_rh,median_vpd)  %>% 
    pivot_wider(names_from = vpd_control, values_from = c(median_rh, median_vpd)) %>% 
    mutate(
        percentage_diff_rh  = (median_rh_no - median_rh_yes) / median_rh_yes * 100,
        percentage_diff_vpd = (median_vpd_no - median_vpd_yes) / median_vpd_yes * 100,
        .keep = "unused"
        )
    
    

In [ ]:
ggsave("./666_figures/1_figure_1_rh_and_vpd.png", 
       plot = (plot_vpd + plot_rh  +  plot_annotation(tag_levels = 'A')), 
       width = 26, height = 12, units = "in", dpi = 1200)